# NOVA AI - Fine-tuning avec Unsloth
**Entraîne ton propre modèle IA gratuitement sur Google Colab**

Ce notebook va :
1. Installer Unsloth (optimisé pour réduire la mémoire de 50%)
2. Charger un modèle open-source (Llama 3.2, Gemma 2, etc.)
3. Préparer TES données (conversations, code, textes)
4. Fine-tuner le modèle
5. Sauvegarder le modèle entraîné

## 1. Installation d'Unsloth

Exécute cette cellule pour installer Unsloth. Cela prend ~2 minutes.

In [ ]:
# Installation d'Unsloth avec support des modèles récents
# Unsloth réduit la mémoire VRAM nécessaire de 50%
# et accélère l'entraînement de 2x

%%capture
!pip install unsloth
!pip install --upgrade --no-deps "git+https://github.com/unslothai/unsloth.git"

print("Unsloth installé avec succès !")

## 2. Choix du modèle de base

Choisis le modèle que tu veux fine-tuner :

| Modèle | Taille | VRAM | Qualité | Recommandé pour |
|--------|--------|------|---------|-----------------|
| `unsloth/Llama-3.2-1B` | 1B | 4GB | Basique | Débuter, tester |
| `unsloth/Llama-3.2-3B` | 3B | 8GB | Moyenne | ✅ Assistant général |
| `unsloth/gemma-2-2b-it` | 2B | 6GB | Moyenne | Chat / dialogue |
| `unsloth/mistral-7b-v0.3` | 7B | 12GB | Haute | ⚠️ Limite Colab gratuit |
| `unsloth/Phi-3-mini-4k` | 3.8B | 6GB | Bonne | Code + raisonnement |

**Pour commencer :** utilise `unsloth/Llama-3.2-3B` (bon équilibre qualité/vitesse sur Colab gratuit).

In [ ]:
from unsloth import FastLanguageModel
import torch

# ====== CONFIGURATION ======
# Change CE MODEL pour choisir ton modèle de base
MODEL_NAME = "unsloth/Llama-3.2-3B"  # ou "unsloth/Llama-3.2-1B", "unsloth/gemma-2-2b-it"

# Paramètres techniques (ne pas toucher si tu débutes)
MAX_SEQ_LENGTH = 2048  # Longueur max des textes d'entraînement
DTYPE = None           # Auto-détection du type de données
LOAD_IN_4BIT = True    # Quantification 4bit (réduit VRAM de 4x)

print(f"Chargement du modèle : {MODEL_NAME}")
print("VRAM disponible:", round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1), "GB")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print("✓ Modèle chargé avec succès !")

## 3. Préparation de TES données

### Format attendu
Chaque exemple doit être au format conversationnel :
```
{
  "instruction": "La question ou instruction",
  "output": "La réponse attendue"
}
```

Tu as **3 options** pour fournir tes données :
- **Option A** : Upload direct (glisse un fichier JSON)
- **Option B** : Google Drive
- **Option C** : JSON en ligne (Hugging Face dataset)

### Option A : Uploader un fichier JSON

Crée un fichier `data.json` avec ce format :

```json
[
  {
    "instruction": "Qu'est-ce que le machine learning ?",
    "output": "Le machine learning est une branche de l'IA qui permet aux ordinateurs d'apprendre à partir de données sans être explicitement programmés."
  },
  {
    "instruction": "Écris une fonction Python pour calculer la factorielle",
    "output": "def factorielle(n):\n    if n <= 1:\n        return 1\n    return n * factorielle(n-1)"
  }
]
```

In [ ]:
from google.colab import files
import json

print("Glisse ton fichier JSON ci-dessous (data.json) :")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
with open(filename, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f"✓ {len(raw_data)} exemples chargés depuis {filename}")
print(f"\nPremier exemple :\n{json.dumps(raw_data[0], indent=2, ensure_ascii=False)}")

### Formatage des données pour l'entraînement

Le modèle doit apprendre avec un format de prompt spécifique.
Pour Llama 3, c'est ce format :
```
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
Question ici<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Réponse ici<|eot_id|>
```

In [ ]:
from datasets import Dataset

# Format de prompt pour Llama 3
def format_prompt(example):
    return {
        "text": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{example['output']}<|eot_id|>"
    }

formatted_data = [format_prompt(ex) for ex in raw_data]
dataset = Dataset.from_list(formatted_data)

print(f"✓ Dataset prêt : {len(dataset)} exemples")
print(f"\nExemple formaté :")
print(dataset[0]['text'][:300] + "...")

## 4. Configuration LoRA (Low-Rank Adaptation)

On n'entraîne pas TOUT le modèle (trop long). On utilise **LoRA** :
on ajoute de petits poids adaptateurs qu'on entraîne seulement.
Résultat : modèle personnalisé en 1-2 heures au lieu de plusieurs jours.

In [ ]:
# Configuration LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                  # Rang LoRA (16 = bon équilibre qualité/vitesse)
    target_modules=[       # Modules à adapter
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,         # Force de l'adaptation
    lora_dropout=0,        # Pas de dropout (meilleur pour petits datasets)
    bias="none",          # Pas de biais
    use_gradient_checkpointing="unsloth",  # Économise la VRAM
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print("✓ Configuration LoRA terminée")
print(f"Paramètres entraînables : {model.get_nb_trainable_parameters()[0]:,} sur {model.get_nb_trainable_parameters()[1]:,}")

## 5. Entraînement !

C'est ici que ça se passe. La durée dépend de :
- Nombre d'exemples
- Taille du modèle
- Colab gratuit GPU T4 : ~30 min pour 100 exemples sur Llama 3.2-3B

**Important :** Colab gratuit se déconnecte après ~2h d'inactivité.
Ne laisse pas le notebook inactif longtemps avant de lancer cette cellule.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,  # True = plus rapide pour petits textes
    args=TrainingArguments(
        per_device_train_batch_size=2,      # Réduire si OOM (Out Of Memory)
        gradient_accumulation_steps=4,       # Simule un batch plus gros
        warmup_steps=5,
        num_train_epochs=3,                  # 3 passages sur les données
        learning_rate=2e-4,                  # Taux d'apprentissage
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",                 # Optimiseur 8bit (économie VRAM)
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",                   # Pas de reporting externe
    ),
)

print("=" * 50)
print("LANCEMENT DE L'ENTRAÎNEMENT...")
print("=" * 50)
print(f"Exemples : {len(dataset)}")
print(f"Époques : 3")
print(f"Modèle : {MODEL_NAME}")
print(f"")

trainer_stats = trainer.train()

print("\n✓ Entraînement terminé !")

## 6. Test du modèle fine-tuné

Avant de sauvegarder, vérifions que le modèle répond correctement.

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Test avec une question de tes données
test_prompt = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n" + raw_data[0]['instruction'] + "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("Question:", raw_data[0]['instruction'])
print("\nRéponse du modèle fine-tuné :")
print("-" * 30)

_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
)
print("\n" + "-" * 30)
print("\n✓ Test terminé")

## 7. Sauvegarde du modèle

Deux options :
- **GGUF** (format léger, pour faire tourner avec Ollama sur ton PC)
- **Hugging Face** (pour partager ou réutiliser plus tard)

### Option recommandée : GGUF + Ollama
Le format GGUF est compressé et peut tourner sur n'importe quel PC.

In [ ]:
# Sauvegarde au format GGUF (pour Ollama)
# Le modèle sera compressé ~4x

print("Export en cours... (2-3 minutes)")
print("")

# Sauvegarder le modèle fine-tuné au format GGUF
model.save_pretrained_gguf(
    "model_gguf",
    tokenizer,
    quantization_method="q4_k_m",  # q4_k_m = bon équilibre qualité/taille
)

print("✓ Export GGUF terminé !")
print("")
print("Taille du fichier :")
!ls -lh model_gguf/

In [ ]:
# Télécharge le modèle sur ton PC
from google.colab import files

print("Téléchargement du modèle GGUF...")
print("Le fichier sera téléchargé dans ton navigateur.")

import os
for f in os.listdir("model_gguf"):
    if f.endswith(".gguf"):
        files.download(os.path.join("model_gguf", f))
        print(f"✓ {f} téléchargé")

## 8. Faire tourner le modèle sur ton PC avec Ollama

Une fois le fichier `.gguf` téléchargé :

### 1. Installe Ollama
Va sur https://ollama.com et télécharge Ollama

### 2. Crée un Modelfile
Crée un fichier `Modelfile` (sans extension) à côté de ton `.gguf` :

```
FROM ./ton_modele.gguf

TEMPLATE """<|begin_of_text|><|start_header_id|>user<|end_header_id|>
{{ .Prompt }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
{{ .Response }}<|eot_id|>"""

PARAMETER temperature 0.7
PARAMETER top_p 0.9
```

### 3. Importe dans Ollama
```bash
ollama create mon-modele -f Modelfile
```

### 4. Utilise-le !
```bash
ollama run mon-modele
```

## Ou option 2 : Sauvegarder sur Hugging Face

In [ ]:
# Option alternative : pousser sur Hugging Face Hub
# (nécessite un compte gratuit sur huggingface.co)

HF_USERNAME = "ton-username"  # Remplace par ton username Hugging Face
MODEL_NAME_HF = "mon-premier-finetuning"  # Nom de ton choix

print("Pour sauvegarder sur Hugging Face :")
print(f"1. Crée un compte sur https://huggingface.co")
print(f"2. Génère un token : Settings → Access Tokens")
print(f"3. Décommente les lignes ci-dessous")
print(f"")

# from huggingface_hub import login, create_repo
# import os

# token = "ton_token_hf"  # Mets ton token ici
# login(token)
# create_repo(f"{HF_USERNAME}/{MODEL_NAME_HF}", exist_ok=True)

# model.save_pretrained(f"{HF_USERNAME}/{MODEL_NAME_HF}")
# tokenizer.save_pretrained(f"{HF_USERNAME}/{MODEL_NAME_HF}")
# print(f"✓ Modèle poussé sur https://huggingface.co/{HF_USERNAME}/{MODEL_NAME_HF}")

---

## Résumé du processus

1. ✅ Exécute les cellules dans l'ordre
2. 📁 Prépare ton fichier `data.json` avec tes exemples
3. 🚨 Lance l'entraînement (ne ferme pas le navigateur !)
4. 💾 Télécharge le modèle GGUF
5. 🖥️ Importe dans Ollama sur ton PC
6. 🎉 Ton modèle fonctionne en local !

### Conseils pour un bon dataset
- **Minimum :** 20-50 exemples
- **Recommandé :** 100-500 exemples
- **Format :** Questions claires + réponses précises
- **Variété :** Plus c'est diversifié, mieux c'est

### Problèmes courants
- **OOM (Out Of Memory)** : mets `per_device_train_batch_size=1`
- **Colab déconnecté** : réexécute tout depuis le début
- **Modèle qui répond mal** : plus de données, ou plus d'époques (5-10)